# KV Cache and Architectural Evolution

> Training predicts many sequence positions in parallel, but autoregressive generation produces tokens one at a time and repeatedly accesses prior context. KV Cache avoids recomputing historical K and V, yet for Llama-7B at a 32K context the cache alone uses about 16 GB of FP16 memory.
>
> This memory cost drove three architectural generations. **MHA** stores separate K/V for every Query head. **MQA and GQA** share K/V heads across several Query heads. **MLA** compresses K/V into a shorter latent representation.
>
> Compression is not an isolated matrix replacement. Head sharing changes Attention's expressiveness, and MLA conflicts with the rotational structure of RoPE. The implementations below check cache size, numerical behavior, and position handling together.

Once a historical token's K and V have been computed from its hidden state, later tokens do not change them. **KV Cache** retains these vectors so each generation step computes only the new position's K/V. The cache itself consumes memory: for 32 layers, 32 heads, head dimension 128, and a 32K context, FP16 KV Cache is about 16 GB.

MQA and GQA reduce the number of stored K/V heads; MLA compresses their information into a smaller latent vector. We begin by examining exactly what the cache stores.


## 1. Repeated Computation in Autoregressive Generation

Suppose four tokens have been generated and the model is producing the fifth:

```text
After step 4: [t1, t2, t3, t4]
Step 5:       [t1, t2, t3, t4, t5?]
              the new query attends to keys and values from t1 through t4
```

The K/V for `t1` comes from its hidden state and projection matrices. In a decoder-only model, `t1` cannot see later tokens, so its hidden state is fixed once calculated. The `k1,v1` from step 4 can be reused at step 5.

Without caching, step 5 recomputes K/V for `t1` through `t4`; by token 100 it recomputes all 99 historical positions. Total work grows quadratically. Storing each K/V once restores linear projection work. The code first verifies the premise: historical K/V values do not change.


In [ ]:
# Verify that historical Tokens really keep the same K when a new Token is generated
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(42)

d_model, d_k = 8, 4
W_K = nn.Linear(d_model, d_k, bias=False)

# End of step 4: hidden states and K have been computed for four Tokens
h_4 = torch.randn(1, 4, d_model)
k_4 = W_K(h_4)

# Step 5: one new Token arrives, making the sequence length five
h_new = torch.randn(1, 1, d_model)
h_5 = torch.cat([h_4, h_new], dim=1)
k_5 = W_K(h_5)  # without caching: recompute the entire sequence

print(f"k_4: {tuple(k_4.shape)}  <- K for four Tokens, already computed last step")
print(f"k_5: {tuple(k_5.shape)}  <- K for five Tokens after full recomputation")
print()
print(f"First four recomputed positions equal cached k_4: {torch.allclose(k_5[:, :4], k_4)}")
print()
# Incremental method: compute only one K for the new Token and concatenate it to history
k_new = W_K(h_new)
k_5_incremental = torch.cat([k_4, k_new], dim=1)
print(f"Incremental computes one item {tuple(k_new.shape)}; concatenated result equals full recomputation: "
      f"{torch.allclose(k_5_incremental, k_5)}")
print()
print("Key observation: recomputing K for historical tokens gives exactly the same result and wastes computation.")
print("Store and reuse history, computing only the new Token each step: that is the entire idea of KV Cache.")


### The Speed–Memory Trade-off of Caching

KV Cache exchanges GPU memory for speed. Each token stores one K/V pair at every layer:

```text
elements per token per layer = 2 x num_kv_heads x head_dim
total elements = seq_len x num_layers x elements above
bytes = total elements x bytes_per_element
```

The factor 2 accounts for K and V. We calculate the full bill for a Llama-7B configuration.


In [ ]:
# Hand-calculate KV Cache usage with concrete Llama-7B numbers

hidden_size = 4096
num_layers = 32
num_heads = 32
head_dim = hidden_size // num_heads  # 128
seq_len = 32768  # 32K context
bytes_per_element = 2  # FP16

# MHA stores num_heads pairs of K and V per Token per layer
elements_per_token_per_layer = num_heads * head_dim * 2  # factor 2 for one K and one V
total_elements = seq_len * num_layers * elements_per_token_per_layer
total_bytes = total_elements * bytes_per_element
total_gb = total_bytes / (1024 ** 3)

print(f"Model: hidden={hidden_size}, layers={num_layers}, heads={num_heads}, head_dim={head_dim}")
print(f"Sequence length: {seq_len}, dtype: bf16")
print()
print(f"K+V elements per Token per layer: {elements_per_token_per_layer}")
print(f"Total KV Cache elements: {total_elements:,}")
print(f"  Total memory:   {total_gb:.1f} GB")
print()
print(f"For comparison, Llama-7B weights use about 13 GB in FP16")
print(f"Key observation: at 32K context, KV Cache already exceeds the model weights")


## 2. MQA and GQA

The cache bill contains `num_heads` as a multiplier. **MQA** (Multi-Query Attention) lets all 32 Query heads share one K/V head, shrinking KV Cache 32-fold. Its cost is reduced diversity because every Query head sees the same K/V representation.

**GQA** (Grouped-Query Attention) is a compromise. It divides Query heads into groups and shares one K/V head within each group. Eight groups for 32 Query heads reduce cache eightfold. More sharing saves memory but may reduce capacity; less sharing approaches MHA.

| Method | `num_kv_heads` | Cache relative to MHA | Example models |
|:---|:---|:---|:---|
| MHA | `num_heads` | 1x | original Transformer, GPT-2 |
| GQA | number of groups | group ratio | Llama 2/3, Qwen, Mistral |
| MQA | 1 | `1/num_heads` | early efficient-decoding designs |

The code shows that GQA stores only the shared copies, then expands them for Query heads during calculation.


In [ ]:
# GQA mechanism: cache only four K heads, then share them among 32 query heads

torch.manual_seed(42)

num_heads = 32
num_kv_heads = 4
head_dim = 128
seq_len = 10

# KV side: cache only four K heads; this is what actually occupies memory
k_cache = torch.randn(1, num_kv_heads, seq_len, head_dim)
print(f"K in cache: {tuple(k_cache.shape)}  <- only {num_kv_heads} copies")

# Q side has 32 heads, each borrowing its group K
# repeat_interleave copies K1 eight times, K2 eight times, and so on
k_expanded = k_cache.repeat_interleave(num_heads // num_kv_heads, dim=1)
print(f"Expanded for Q: {tuple(k_expanded.shape)}  <- {num_heads} copies, but only four distinct values")
print()
print("Key observation: computation still uses 32 heads, but the cache stores only four KV groups;")
print("GQA saves storage, not the number of query heads.")


In [ ]:
# Compare KV Cache bills for three Attention variants under the Llama-7B configuration


def kv_cache_gb(num_heads, head_dim, num_layers, seq_len, num_kv_heads):
    """Calculate KV Cache size in GB."""
    per_token_per_layer = num_kv_heads * head_dim * 2  # K + V
    total = seq_len * num_layers * per_token_per_layer
    return total * 2 / (1024 ** 3)  # FP16


num_heads = 32
head_dim = 128
num_layers = 32
seq_len = 32768

configs = [
    ("MHA", num_heads),
    ("GQA-8", num_heads // 8),  # eight groups -> four KV heads
    ("GQA-4", num_heads // 4),  # four groups -> eight KV heads
    ("MQA", 1),
]

print(f"{'Scheme':<8}{'KV heads':<8}{'GB':<10}{'vs MHA'}")
print("-" * 40)
base = None
for name, n_kv in configs:
    gb = kv_cache_gb(num_heads, head_dim, num_layers, seq_len, n_kv)
    if base is None:
        base = gb
    print(f"{name:<8}{n_kv:<8}{gb:<10.2f}{gb / base:.2f}×")

print()
print("Key observation: MQA shrinks cache to one 32nd, but sharing one K/V pair across all heads visibly hurts quality.")
print("GQA is the common engineering compromise: Llama 2 70B uses eight groups, and Llama 3 8B uses four.")


### Limits of MQA and GQA

MQA is the maximum possible sharing along the head-count dimension. Compressing further by removing more K/V heads would sacrifice additional Attention diversity.

MLA asks a different question: rather than storing projected K and V themselves, can we store a smaller intermediate representation and reconstruct K/V when needed?


## 3. Multi-Head Latent Attention

Standard Attention projects hidden state `h` into `K = W_K h` and `V = W_V h`. **MLA** (Multi-head Latent Attention, introduced by DeepSeek-V2) factors this into compression and reconstruction:

```text
c = W_DK h     # compress h into a d_c-dimensional latent vector
K = W_UK c     # reconstruct K
V = W_UV c     # reconstruct V
```

Inference stores only `c`; K and V are reconstructed on demand. Because `d_c` is much smaller than `num_heads x head_dim`, cache size falls sharply. The compression and reconstruction matrices are learned, allowing training to preserve the information that Attention needs and discard redundancy. This differs from GQA, which removes copies after projection. We first calculate the compression ratio for DeepSeek-V2 with `d_c=512`, 128 heads, and head dimension 128.


In [ ]:
# Hand-calculate MLA compression with concrete numbers

# DeepSeek-V2-scale Attention configuration
num_heads = 128
head_dim = 128
mha_per_token_per_layer = num_heads * head_dim * 2  # one K and one V

# MLA shares one d_c-dimensional latent between K and V
d_c = 512
mla_per_token_per_layer = d_c

print(f"MHA per Token per layer: {mha_per_token_per_layer} elements")
print(f"  = {num_heads} heads × {head_dim} dim × 2 (K+V)")
print()
print(f"MLA per Token per layer: {mla_per_token_per_layer} elements")
print(f"  = d_c = {d_c}, one latent shared by K and V")
print()
print(f"Compression: {mha_per_token_per_layer / mla_per_token_per_layer:.0f}x")
print(f"Saved: {(1 - mla_per_token_per_layer / mha_per_token_per_layer) * 100:.1f}%")
print()
print("Key observation: GQA saves a constant factor from grouping; MLA changes the storage scale.")


## 4. Implementing a Simplified MLA from Scratch

The teaching implementation separates two paths:

- **Q path**: standard `W_Q` projection into `num_heads` Query heads.
- **KV path**: `W_DK` compresses to a `d_c` latent, then `W_UK` and `W_UV` reconstruct K and V.

RoPE is omitted here so the main mechanism remains visible; Section 6 handles its conflict with latent compression.


In [ ]:
# Simplified MLA: the core is latent compression


class SimpleMLA(nn.Module):
    """Teaching MLA implementation showing latent compression.

    Parameters:
        d_model: model hidden dimension
        num_heads: number of query heads
        head_dim: dimension of each head
        d_c: latent dimension shared by K and V
    """

    def __init__(self, d_model, num_heads, head_dim, d_c):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = head_dim
        self.d_c = d_c

        # Q path: standard projection, just like ordinary Attention
        self.W_Q = nn.Linear(d_model, num_heads * head_dim, bias=False)

        # KV path: compress into latent c first, then reconstruct K and V
        self.W_DK = nn.Linear(d_model, d_c, bias=False)               # down
        self.W_UK = nn.Linear(d_c, num_heads * head_dim, bias=False)  # up to K
        self.W_UV = nn.Linear(d_c, num_heads * head_dim, bias=False)  # up to V

        # Output projection
        self.W_O = nn.Linear(num_heads * head_dim, d_model, bias=False)

    def forward(self, x):
        """x: [batch, seq_len, d_model]; return (output, latent c)."""
        B, S, D = x.shape

        # Q path
        Q = self.W_Q(x).view(B, S, self.num_heads, self.head_dim)

        # KV path: c is the latent stored in the cache
        c = self.W_DK(x)                                          # [B, S, d_c]
        K = self.W_UK(c).view(B, S, self.num_heads, self.head_dim)
        V = self.W_UV(c).view(B, S, self.num_heads, self.head_dim)

        # Standard Attention calculation, without a mask in this teaching version
        Q = Q.transpose(1, 2)  # [B, heads, S, head_dim]
        K = K.transpose(1, 2)
        V = V.transpose(1, 2)
        scores = Q @ K.transpose(-1, -2) / (self.head_dim ** 0.5)
        attn = scores.softmax(dim=-1)
        out = attn @ V

        out = out.transpose(1, 2).reshape(B, S, self.num_heads * self.head_dim)
        return self.W_O(out), c  # also return c so the cache can be inspected


# Test with a small configuration
torch.manual_seed(42)
d_model, num_heads, head_dim, d_c = 64, 4, 16, 16

mla = SimpleMLA(d_model, num_heads, head_dim, d_c)
x = torch.randn(1, 10, d_model)
out, c = mla(x)

print(f"Input shape: {tuple(x.shape)}")
print(f"Output shape: {tuple(out.shape)}")
print(f"Latent c shape: {tuple(c.shape)}  <- everything stored in the cache")
print()
mha_elems = num_heads * head_dim * 2
print(f"Standard MHA caches K+V = {mha_elems} elements per Token per layer")
print(f"MLA caches only c = {d_c} elements per Token per layer")
print(f"Compression: {mha_elems / d_c:.0f}x")


## 5. Comparing Three KV Cache Designs

Plot MHA, GQA, and MLA cache growth against context length using a DeepSeek-V2-scale configuration: `num_layers=60`, `num_heads=128`, `head_dim=128`, and `d_c=512`.


In [ ]:
# Curves showing cache growth with context length for three generations of designs


def mha_cache_gb(num_heads, head_dim, num_layers, seq_len):
    """KV Cache size in GB for standard MHA."""
    return seq_len * num_layers * num_heads * head_dim * 2 * 2 / (1024 ** 3)


def gqa_cache_gb(num_heads, head_dim, num_layers, seq_len, num_groups):
    """KV Cache size in GB for GQA with num_groups sharing K/V."""
    num_kv_heads = num_heads // num_groups
    return seq_len * num_layers * num_kv_heads * head_dim * 2 * 2 / (1024 ** 3)


def mla_cache_gb(d_c, num_layers, seq_len):
    """KV Cache size in GB for MLA, storing only latent; the RoPE bypass is added in Section 6."""
    return seq_len * num_layers * d_c * 2 / (1024 ** 3)


# DeepSeek-V2-scale configuration
num_layers = 60
num_heads = 128
head_dim = 128
d_c = 512

seq_lens = [4096, 8192, 16384, 32768, 65536, 131072]
mha_sizes = [mha_cache_gb(num_heads, head_dim, num_layers, s) for s in seq_lens]
gqa_sizes = [gqa_cache_gb(num_heads, head_dim, num_layers, s, 8) for s in seq_lens]
mla_sizes = [mla_cache_gb(d_c, num_layers, s) for s in seq_lens]

plt.figure(figsize=(10, 5))
plt.plot(seq_lens, mha_sizes, 'o-', label=f'MHA (heads={num_heads})', linewidth=2)
plt.plot(seq_lens, gqa_sizes, 's-', label=f'GQA-8 (kv_heads={num_heads // 8})', linewidth=2)
plt.plot(seq_lens, mla_sizes, '^-', label=f'MLA (d_c={d_c})', linewidth=2)
plt.xlabel('Context Length (tokens)')
plt.ylabel('KV Cache (GB, FP16)')
plt.title('KV cache growth with context length')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xscale('log', base=2)
plt.xticks(seq_lens, [f'{s // 1024}K' for s in seq_lens])
plt.tight_layout()
plt.show()

print("KV Cache at 128K context:")
print(f"  MHA:   {mha_sizes[-1]:7.1f} GB")
print(f"  GQA-8: {gqa_sizes[-1]:7.1f} GB  ({mha_sizes[-1] / gqa_sizes[-1]:.0f}x compression)")
print(f"  MLA:   {mla_sizes[-1]:7.1f} GB  ({mha_sizes[-1] / mla_sizes[-1]:.0f}x compression)")
print()
print("Key observation: at long context lengths, MLA saves another order of magnitude beyond GQA.")


## 6. MLA and RoPE

RoPE rotates Q and K so their dot product depends on relative position. MLA wants to avoid storing K, but RoPE needs K before it can apply a position-dependent rotation. The rotation cannot be absorbed into one fixed projection matrix because its angle changes with position.

DeepSeek-V2 uses **decoupled RoPE**. Content follows the compressed latent path, while position uses a small side path:

```text
content path: c = W_DK h                 -> cache
position path: k_rope = RoPE(W_KR h)     -> cache (small)
final K per head = concat(content reconstructed from c, k_rope)
```

The `k_rope` vector is shared by all heads and has dimension `d_r=64`, rather than one copy per head. The real MLA cache therefore stores `d_c + d_r = 512 + 64 = 576` elements per token per layer.


In [ ]:
# Full MLA bill: latent plus RoPE bypass under the DeepSeek-V2 configuration

num_heads = 128
head_dim = 128
d_c = 512  # latent dimension
d_r = 64   # RoPE bypass dimension, one shared copy across all heads

mha_per_token_per_layer = num_heads * head_dim * 2   # one K and one V
mla_per_token_per_layer = d_c + d_r                  # latent plus shared RoPE vector

print(f"MHA per Token per layer: {mha_per_token_per_layer:6d} elements")
print(f"MLA per Token per layer: {mla_per_token_per_layer:6d} elements"
      f" (latent {d_c} + RoPE bypass {d_r})")
print()
print(f"Compression: {mha_per_token_per_layer / mla_per_token_per_layer:.0f}x")
print()
print("Key observation: the RoPE bypass is only 64 shared dimensions, so it barely reduces the compression benefit.")
print("If the bypass stored one copy per head (128×64), the bill would expand to 8,704; sharing is crucial.")


## 7. MLA in Production

MLA was deployed at scale in DeepSeek-V2, retained by DeepSeek-V3, and adopted by Kimi K2. A cache an order of magnitude smaller allows longer contexts or more concurrent requests in the same memory. Since generation is often bandwidth-bound, reading a smaller cache may also accelerate long-context decoding.

The trade-off is extra matrix multiplication for compression and reconstruction, plus newer and less mature support in inference engines such as vLLM, SGLang, and TensorRT-LLM.

In one sentence: **GQA compresses the number of copies; MLA compresses their content.** The first saves a constant factor, while the second can save an order of magnitude.


## Summary

- [ ] The KV Cache is the K and V of all historical tokens stored by a decoder-only model during inference, growing linearly with context length
- [ ] Under 32K context, a 7B model's KV Cache is about 16 GB, already close to the size of the model weights themselves
- [ ] GQA / MQA compress the cache by reducing the number of KV heads, at the cost of reduced attention diversity
- [ ] MLA jointly projects K and V into a low-dimensional latent, caching only the latent during inference; its compression ratio is far higher than GQA
- [ ] During inference MLA caches only the latent `c` and projects back to K and V when needed; the mechanism is an information bottleneck
- [ ] RoPE cannot be applied directly on top of MLA; DeepSeek solves this with decoupled RoPE — the main part goes through the latent, a small part goes through the RoPE side path
- [ ] DeepSeek-V2 / V3 / Kimi K2 all adopt MLA; the cache is compressed by an order of magnitude, and inference is faster in long-context scenarios

References: [DeepSeek-V2](https://arxiv.org/abs/2405.04434), [DeepSeek-V3](https://arxiv.org/abs/2412.19437), [Kimi K2](https://arxiv.org/abs/2507.20534), [GQA paper](https://arxiv.org/abs/2305.13245).


## Exercises

> You may ask AI for help explaining the ideas, but it is not recommended to let AI "solve this problem for you" directly.

**Exercise 1: KV Cache Size Calculation**

Given the model configuration: `hidden_size=4096, num_layers=32, num_heads=32, head_dim=128, seq_len=16384, dtype=FP16`. Compute the KV Cache footprint under MHA in GB.

Hint: K+V elements per token per layer = `num_heads × head_dim × 2`. Total elements = `seq_len × num_layers × that value`. Bytes = elements × 2. GB = bytes / 1024³.


**Exercise 1: Calculate KV Cache Size**

For `hidden_size=4096`, `num_layers=32`, `num_heads=32`, `seq_len=16384`, and FP16, calculate MHA KV Cache size in GB.

Hint: per token per layer, K+V elements are `num_heads x head_dim x 2`; multiply by sequence length and layers, then by 2 bytes and divide by $1024^3$.


In [ ]:
# Exercise 1: KV Cache capacity calculation

hidden_size = 4096
num_layers = 32
num_heads = 32
head_dim = hidden_size // num_heads
seq_len = 16384
bytes_per_element = 2  # FP16

# TODO: Replace the triple-quoted content below with your code
elements_per_token_per_layer = 'TODO: replace this placeholder with your code'
total_elements = 'TODO: replace this placeholder with your code'
total_gb = 'TODO: replace this placeholder with your code'

assert not isinstance(elements_per_token_per_layer, str), 'Please replace the placeholder before running the assertion.'
assert not isinstance(total_elements, str), 'Please replace the placeholder before running the assertion.'
assert not isinstance(total_gb, str), 'Please replace the placeholder before running the assertion.'

expected_eptpl = num_heads * head_dim * 2
expected_total = seq_len * num_layers * expected_eptpl
expected_gb = expected_total * bytes_per_element / (1024 ** 3)

assert elements_per_token_per_layer == expected_eptpl,     f"Per Token per layer should be {expected_eptpl}; you got {elements_per_token_per_layer}"
assert total_elements == expected_total,     f"Total elements should be {expected_total}; you got {total_elements}"
assert abs(total_gb - expected_gb) < 0.01, f"Should be {expected_gb:.2f} GB; you got {total_gb}"

print("Exercise 1 passed")
print(f"   K+V per Token per layer: {elements_per_token_per_layer} elements")
print(f"   total elements: {total_elements:,}")
print(f"  Total memory:   {total_gb:.1f} GB")


**Exercise 2: Calculate GQA Cache Size**

For `num_heads=32`, `head_dim=128`, `num_groups=8`, `num_layers=32`, `seq_len=32768`, and FP16, calculate GQA-8 KV Cache size.

Hint: the number of KV heads is `num_heads // num_groups`; multiply it by `head_dim x 2` per token per layer.


In [ ]:
# Exercise 2: cache size after GQA grouping

num_heads = 32
head_dim = 128
num_groups = 8
num_layers = 32
seq_len = 32768
bytes_per_element = 2  # FP16

# TODO: Replace the triple-quoted content below with your code
num_kv_heads = 'TODO: replace this placeholder with your code'
total_gb = 'TODO: replace this placeholder with your code'

assert not isinstance(num_kv_heads, str), 'Please replace the placeholder before running the assertion.'
assert not isinstance(total_gb, str), 'Please replace the placeholder before running the assertion.'

expected_kv_heads = num_heads // num_groups
expected_eptpl = expected_kv_heads * head_dim * 2
expected_gb = seq_len * num_layers * expected_eptpl * bytes_per_element / (1024 ** 3)

assert num_kv_heads == expected_kv_heads,     f"KV-head count should be {expected_kv_heads}; you got {num_kv_heads}"
assert abs(total_gb - expected_gb) < 0.01,     f"Should be {expected_gb:.2f} GB; you got {total_gb}"

print("Exercise 2 passed")
print(f"   actual KV heads: {num_kv_heads}, shared in {num_groups} groups")
print(f"   GQA-8 KV Cache: {total_gb:.2f} GB; MHA is 16 GB under the same configuration")


**Exercise 3: Complete the forward of the Simplified MLA**

The `SimpleMLA` class below omits the latent c computation and the K and V projection steps in forward. Fill in these three lines so the asserts pass.

Hint: refer to the implementation in Section 4. `c = self.W_DK(x)`, then `K = self.W_UK(c).view(...).transpose(1, 2)`, and V similarly.


In [ ]:
# Exercise 3: complete the simplified MLA forward pass


class SimpleMLA(nn.Module):
    def __init__(self, d_model, num_heads, head_dim, d_c):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = head_dim
        self.d_c = d_c
        self.W_Q = nn.Linear(d_model, num_heads * head_dim, bias=False)
        self.W_DK = nn.Linear(d_model, d_c, bias=False)
        self.W_UK = nn.Linear(d_c, num_heads * head_dim, bias=False)
        self.W_UV = nn.Linear(d_c, num_heads * head_dim, bias=False)
        self.W_O = nn.Linear(num_heads * head_dim, d_model, bias=False)

    def forward(self, x):
        B, S, D = x.shape
        Q = self.W_Q(x).view(B, S, self.num_heads, self.head_dim).transpose(1, 2)

        # TODO: complete the following three lines
        c = None   # compress to latent
        K = None   # reconstruct K and view as heads, then transpose to [B, heads, S, head_dim]
        V = None   # reconstruct V in the same way

        assert c is not None and K is not None and V is not None, 'Please replace the placeholder before running the assertion.'

        scores = Q @ K.transpose(-1, -2) / (self.head_dim ** 0.5)
        attn = scores.softmax(dim=-1)
        out = attn @ V
        out = out.transpose(1, 2).reshape(B, S, self.num_heads * self.head_dim)
        return self.W_O(out), c


# Verify
torch.manual_seed(42)
mla_hw = SimpleMLA(d_model=64, num_heads=4, head_dim=16, d_c=16)
x = torch.randn(1, 10, 64)
out, c = mla_hw(x)

assert out.shape == (1, 10, 64), f"Output shape should be (1, 10, 64); got {tuple(out.shape)}"
assert c.shape == (1, 10, 16), f"Latent shape should be (1, 10, 16); got {tuple(c.shape)}"
assert torch.isfinite(out).all(), "Output contains NaN or Inf; check the K/V projections"

print("Exercise 3 passed")
print(f"   output shape: {tuple(out.shape)}, latent shape: {tuple(c.shape)}")
print("   MLA's core is latent compression: cache only the d_c-dimensional c per Token per layer")
